# DebateBot: Two Sides and a Judge — Backend Notebook

**Project:** Two agents argue opposing sides of a user-chosen topic across several rounds, while a judge agent summarizes the strongest points and declares a reasoned outcome.

**Stack:** LangGraph (multi-agent orchestration) + a real web-search evidence tool (DuckDuckGo) + a judge agent.

This notebook develops and tests the **backend logic**, which lives in `debate_backend.py` (imported below) so the exact same code also powers the Streamlit frontend in `app.py`. Run cells top to bottom. Works the same on Windows, Mac, and Linux — no hardcoded paths anywhere.


## 1. Install dependencies
Run once. (If you already ran `pip install -r requirements.txt` in your terminal, this is a no-op.)

In [1]:
!pip install -q langgraph langchain-groq langchain-core ddgs grandalf python-dotenv

## 2. Import the backend

`debate_backend.py` must be in the same folder as this notebook. Importing it automatically loads a local `.env` file (if you made one) via `python-dotenv`, so your API key gets picked up here without any extra steps — on any OS.

The module contains:
- `evidence_search(query)` — the evidence tool (live DuckDuckGo web search)
- `DebateState` — the shared state schema passed between graph nodes
- `proponent_node` / `opponent_node` — debater agents pinned to fixed sides
- `judge_node` — the impartial judge agent
- `build_debate_graph()` — assembles everything into a compiled LangGraph
- `run_debate(topic, max_rounds)` — one-shot runner
- `stream_debate(topic, max_rounds)` — generator that yields state after every node (used by the Streamlit UI to render turns live)


In [2]:
from debate_backend import (
    evidence_search,
    build_debate_graph,
    run_debate,
    stream_debate,
)


## 3. Set your API key (if not already picked up)

If you created a `.env` file in this folder with:
```
GROQ_API_KEY=your-real-key-here
```
...then step 2 already loaded it and this cell will do nothing. Otherwise it asks you to paste your key for this session only (hidden input, nothing saved to disk).


In [3]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

# Optional: change the model used by both debaters and the judge
os.environ.setdefault("GROQ_MODEL", "llama-3.3-70b-versatile")


'llama-3.3-70b-versatile'

## 4. Sanity-check the evidence tool on its own
This is the 'evidence tool' referenced in the project brief — a real web search the debaters must cite from, so arguments cite sources instead of just asserting claims.

In [4]:
results = evidence_search("benefits of banning smartphones in schools", max_results=3)
for r in results:
    print(r["title"])
    print(r["snippet"])
    print(r["url"])
    print("-" * 60)


Banning Smartphones in Schools: Review of the Literature Shows Positive ...
Smartphones have become pervasive distractions in schools. More than 95 percent of U.S. teens own devices, with nearly half reporting excessive social media use. Research shows that cell phone bans can improve academic achievement. Evidence on mental health benefits remains mixed, though schools have reported fewer behavioral problems, reduced bullying, and improved classroom engagement. U.S ...
https://paragoninstitute.org/public-health/banning-smartphones-in-schools/
------------------------------------------------------------
Why Should Mobile Phones Be Banned in Schools? 7 Reasons
Discover why mobile phones should be banned in schools. Learn 7 research-backed reasons and how schools can create effective phone-free.
https://wiserread.com/why-should-mobile-phones-be-banned-in-schools/
------------------------------------------------------------
PDF Banning Smartphones in Schools - paragoninstitute.org
Smartph

## 5. Inspect the graph structure

LangGraph makes the control flow explicit: `proponent -> opponent -> advance_round -> (loop back to proponent, or go to judge once max_rounds is reached) -> END`.


In [5]:
graph = build_debate_graph()
print(graph.get_graph().draw_ascii())


          +-----------+      
          | __start__ |      
          +-----------+      
                 *           
                 *           
                 *           
          +-----------+      
          | proponent |      
          +-----------+      
           **         ..     
         **             ..   
        *                 .. 
+----------+                .
| opponent |              .. 
+----------+            ..   
           **         ..     
             **     ..       
               *   .         
        +---------------+    
        | advance_round |    
        +---------------+    
                 .           
                 .           
                 .           
            +-------+        
            | judge |        
            +-------+        
                 *           
                 *           
                 *           
           +---------+       
           | __end__ |       
           +---------+       


## 6. Run a full debate (one-shot)
This calls the graph end-to-end and returns only the final state — good for quick tests.

In [6]:
final_state = run_debate(
    topic="Should schools ban smartphones for students under 16?",
    max_rounds=2,
)

for turn in final_state["transcript"]:
    print(f"[Round {turn['round']}] {turn['side']}:")
    print(turn['argument'])
    print("Sources:", [s['url'] for s in turn['sources'] if s['url']])
    print()

print("=" * 70)
print("JUDGE'S VERDICT")
print("=" * 70)
print(final_state["verdict"])


[Round 1] Proponent:
As we consider the impact of smartphones on students under 16, it's essential to examine the existing research on the topic. According to the study "To Ban or Not to Ban? A Rapid Review on the Impact of Smartphone Bans in Schools on Social Well-Being and Academic Performance" (https://www.mdpi.com/2227-7102/14/8/906), restricting smartphone use in schools can lead to better academic outcomes by reducing distractions and improving student focus and engagement. Since there are no opposing arguments yet, I'd like to emphasize that the available data suggests a positive correlation between smartphone bans and student performance, as also supported by the "Banning Smartphones in Schools: Review of the Literature Shows Positive Impact" (https://paragoninstitute.org/public-health/banning-smartphones-in-schools/), which found significant improvements in student performance, particularly for low-achieving or disadvantaged students. By banning smartphones, schools can create

## 7. Stream a debate turn-by-turn

This is the same generator the Streamlit frontend (`app.py`) consumes to render each argument as it's produced, instead of waiting for the whole debate to finish. Useful here for debugging round-by-round.


In [7]:
for update in stream_debate("Is remote work better than office work for software teams?", max_rounds=2):
    for node_name, partial_state in update.items():
        if node_name in ("proponent", "opponent") and "transcript" in partial_state:
            turn = partial_state["transcript"][-1]
            print(f"--- {turn['side']} (Round {turn['round']}) ---")
            print(turn["argument"])
            print()
        if node_name == "judge":
            print("=== VERDICT ===")
            print(partial_state["verdict"])


--- Proponent (Round 1) ---
As we begin this debate, I'd like to highlight the numerous benefits of remote work for software teams, backed by concrete data and research. According to the "Remote Work Productivity Study: Surprising Findings From a 4-Year ..." by Great Place to Work, remote work has been shown to have a positive impact on productivity, with performance remaining strong even after an extended period of working from home. Additionally, a systematic review of 50+ studies by eMonitor, as seen in their "Remote Work Productivity Study 2026", provides real numbers on remote vs hybrid output, further solidifying the case for remote work. This wealth of data-driven insights, including measurable productivity gains seen in the "Enhancing Remote Workforce Productivity with Microsoft Teams" study by IJIRT, demonstrates that remote work can be an effective and productive arrangement for software teams.

--- Opponent (Round 1) ---
While the proponent cites studies showing positive imp

## 8. Where the frontend fits in

`app.py` (Streamlit) imports `stream_debate` from this same `debate_backend.py` module — no logic is duplicated. To launch the actual UI:

```bash
streamlit run app.py
```

It gives you: a sidebar to enter your API key, pick a model and number of rounds, a topic box, and a live chat-style transcript with expandable sources per turn, ending in the judge's verdict.
